In [1]:
# =====================================================================
#  ライブラリのインストール
# =====================================================================

%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =====================================================================
#  インポート
# =====================================================================

from pathlib import Path                                    # ファイルパスを扱うためのライブラリ
import pandas as pd                                         # データフレームを扱うためのライブラリ                                               
import numpy as np                                          # 数値計算を行うためのライブラリ            
from PIL import Image                                       # 画像処理を行うためのライブラリ
import collections                                          # データの分布をカウントするためのライブラリ
from sklearn.model_selection import train_test_split        # データセットを分割するためのライブラリ
import torch                                                # PyTorchのメインライブラリ
import torch.nn as nn                                       # ニューラルネットワークのモジュールを提供するライブラリ
from torchvision import transforms                          # 画像の前処理を行うためのライブラリ
from torch.utils.data import Dataset                        # データセットを作成するためのライブラリ
from torch.utils.data import DataLoader                     # データローダーを作成するためのライブラリ    
from torchvision import models                              # 事前学習済みモデルを提供するライブラリ

# おおまかなプロセス


1. データ理解

        画像サイズ、カラー or モノクロ、クラス数、データ数を確認

2. 前処理

        画像サイズ統一、数値スケール

3. データ分割

4. モデル準備

5. 学習

6. 評価

7. 改善


# ResNetを採用する理由

   初めはEfficientNetを採用する予定だったが、今回は画素数とモノクロとシンプルなデータ構造なので

   扱いやすいResNetに変更。

   ResNetはスキップ結合（残差接続）のF(x) + x（入力の値）という出力で、

   層が深くなっても勾配消失が起こりにくい。

# 1. データ理解

## 画像サイズ

In [3]:
# from collections import Counter

# # =============================
# # 画像のピクセルサイズを調べる
# # =============================
# def check_image_sizes(paths):
#     """
#     リサイズ前の元画像のサイズ(width, height)を収集して集計する
#     """
#     sizes = []
#     for path in paths:
#         img = Image.open(path)
#         sizes.append(img.size)  # (width, height) のタプル
#     return sizes

# # train / test それぞれ調べる
# train_sizes = check_image_sizes(train_paths)
# test_sizes  = check_image_sizes(test_paths)

# # =============================
# # 集計して表示
# # =============================
# def print_size_summary(sizes, label=""):
#     counter = Counter(sizes)
#     print(f"\n{'='*40}")
#     print(f"{label} 画像サイズの種類: {len(counter)} 種類")
#     print(f"{'='*40}")
#     for size, count in counter.most_common():
#         print(f"  {size[0]}x{size[1]} px  →  {count} 枚")

# print_size_summary(train_sizes, label="Train")
# print_size_summary(test_sizes,  label="Test")

[結果] 28 × 28ピクセルの画像のみ

## 画像の色種類

In [4]:
# # =============================
# # 画像がモノクロかどうか調べる
# # =============================
# def check_image_modes(paths):
#     """
#     画像のモード(RGB, L, RGBAなど)を収集して集計する
#     """
#     modes = []
#     for path in paths:
#         img = Image.open(path)
#         modes.append(img.mode)  # "L"=グレースケール, "RGB"=カラー など
#     return modes

# # train / test それぞれ調べる
# train_modes = check_image_modes(train_paths)
# test_modes  = check_image_modes(test_paths)

# # =============================
# # 集計して表示
# # =============================
# def print_mode_summary(modes, label=""):
#     counter = Counter(modes)
#     print(f"\n{'='*40}")
#     print(f"{label} 画像モードの種類: {len(counter)} 種類")
#     print(f"{'='*40}")
#     for mode, count in counter.most_common():
#         print(f"  {mode}  →  {count} 枚")
    
#     # モノクロのみかどうか判定
#     is_grayscale_only = all(m == "L" for m in modes)
#     print(f"\n  → モノクロ(L)のみ: {'✅ Yes' if is_grayscale_only else '❌ No（カラー混在）'}")

# print_mode_summary(train_modes, label="Train")
# print_mode_summary(test_modes,  label="Test")

[結果] モノクロのみ

## 1. 画像ファイルのパス一覧と正解ラベルデータの読み込み

In [5]:
# 画像のパスを再帰的に収集する関数
def collect_image_paths(root_dir, extension=".jpg"):
    root = Path(root_dir)
    return sorted(root.rglob(f"*{extension}"))

# 学習画像のパスを収集
train_paths = collect_image_paths("../data/train/")

# 学習データの正解ラベルを読み込む
train_df = pd.read_csv("../data/train_master.csv")

labels = train_df["category_id"].values

# テスト画像のパスを収集
test_paths  = collect_image_paths("../data/test/")

# テスト画像のパスを数値順に並び替える
test_paths = sorted(
    collect_image_paths("../data/test/"),
    key=lambda p: int(p.stem.split("_")[1])
)

# 確認
for p in test_paths[:5]:
    print(p.name)

# 学習時にラベルデータと画像データの並びを一致させる必要があるので
# 事前に並べ替えを行う

# ファイル名をキーにした辞書を作成
path_dict = {p.name: p for p in collect_image_paths("../data/train/")}

# CSVのファイル名順にパスとラベルを並び替える
train_paths = [path_dict[fname] for fname in train_df["file_name"]]
labels      = train_df["category_id"].values

# 確認
for i in range(5):
    print(f"パス: {train_paths[i].name}  ラベル: {labels[i]}")

print(f"train画像数: {len(train_paths)}")
print(f"test画像数 : {len(test_paths)}")
print(f"ラベル数   : {len(labels)}")

test_0.jpg
test_1.jpg
test_2.jpg
test_3.jpg
test_4.jpg
パス: train_0.jpg  ラベル: 5
パス: train_1.jpg  ラベル: 0
パス: train_2.jpg  ラベル: 4
パス: train_3.jpg  ラベル: 1
パス: train_4.jpg  ラベル: 9
train画像数: 60000
test画像数 : 10000
ラベル数   : 60000


    def collect_image_paths(root_dir, extension=".jpg"):

⇒root_dir : 探索したいフォルダのパスを受け取る引数

extension=".jpg" : 探すファイルの拡張子。

デフォルトで .jpg を指定しているので、呼び出す時に省略できる

    root = Path(root_dir)

⇒文字列のパス（"../data/train/"）を Pathオブジェクト に変換している

    return sorted(root.rglob(f"*{extension}"))

⇒rglob() はフォルダの中を 再帰的（サブフォルダの中まで） に検索する関数

f"*{extension}" は "*.jpg" という意味で、「.jpgで終わる全てのファイル」を指す

    train_paths = [path_dict[fname] for fname in train_df["file_name"]]

⇒csvのファイル名で辞書かした学習データのパスを検索し、ヒットしたパスをtrain_pathsに一つずつ格納していく

## 2. Transformの定義

In [6]:
IMG_SIZE = 224

# 学習データの前処理は、リサイズ、グレースケール変換、データ拡張、テンソル化、正規化を行う
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),            # リサイズ画像サイズ定義
    transforms.Grayscale(num_output_channels=3),        # グレースケールかつ3チャンネルに変換
    transforms.RandomRotation(10),                      # ランダムに-10度から+10度の範囲で回転
    transforms.RandomAffine(0, translate=(0.1, 0.1)),   # 画像をランダムに上下左右に平行移動（最大10%）させる
    transforms.ToTensor(),                              # テンソル化
    transforms.Normalize(                               # 正規化
        mean=[0.485, 0.456, 0.406],                     # ImageNetの平均値
        std=[0.229, 0.224, 0.225]                       # ImageNetの標準偏差
    ),
])

# テストデータの前処理は、データ拡張を行わず、リサイズと正規化のみを行う
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

        transforms.Compose()

⇒リストの中の処理を 上から順番に 画像に適用する仕組み

        transforms.Resize((IMG_SIZE, IMG_SIZE)),

⇒ResNetは224 * 224ピクセルの画像で扱うことが推奨されているので、224 * 224にリサイズ
        
なお、今回は28 * 28なので引き延ばされてぼやけるが文字の形は保たれるので問題なし。

もし、大きい画像を224 * 224に縮小する場合は３つ手法がある

方法1. ランダムクロップ（最も一般的）

        256などすこし大きいサイズにリサイズして、そこからランダムに224 * 224を切り出す

        画像の一部を切り出すので、ピクセルの情報を損なわずに済む

方法2. タイル分割（医療画像や衛星画像など文字や細部が重要な場合）

        複数の224 * 224に分割してそれぞれ推論する

方法3. 大きい入力サイズのモデルを使う

        # EfficientNet-B7は600×600まで対応

        model = models.efficientnet_b7(weights="IMAGENET1K_V1")

        # Vision Transformerは任意サイズに対応しやすい

        model = models.vit_b_16(weights="IMAGENET1K_V1")

        transforms.Grayscale(num_output_channels=3)

⇒モノクロ画像を3チャンネルに変換する

ResNetはRGB（3チャンネル）を前提としているため、モノクロのままでは入力できない

num_output_channels=3 で同じ画像を3枚重ねて3チャンネルとして扱う

        transforms.RandomRotation(10),

⇒画像をランダムに ±10度 回転させる

手書き文字は人によって傾きが違うため、それを学習に活かすデータ拡張

        transforms.RandomAffine(0, translate=(0.1, 0.1))

⇒画像をランダムに 上下左右に最大10% ずらす

文字の位置がズレても認識できるようにするデータ拡張

        transforms.ToTensor()

⇒PIL画像を PyTorchが扱える Tensor形式 に変換する

あわせてピクセル値を 0〜255 から 0.0〜1.0 に正規化する

        transforms.Normalize(mean, std)

⇒ImageNetの平均・標準偏差でさらに正規化する

ResNetはImageNetで事前学習されているため、同じ基準に揃える必要がある

値はImageNetで公開されている固定値

## 3. Datasetクラスの定義

In [7]:
class HandwritingDataset(Dataset):

    # イニシャライズ
    def __init__(self, paths, labels=None, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    # データセットのサイズを返す
    def __len__(self):
        return len(self.paths)

    # データセットから1サンプルを取得する
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")  # idx番目の画像を開く L = グレースケール RGB = カラー
        if self.transform:
            img = self.transform(img)                   # Transformを適用
        if self.labels is not None:
            return img, self.labels[idx]                # trainはimg+ラベルを返す
        return img                                      # testはimgのみ返す 

PyTorchに「どうやって画像を1枚取り出すか」を教えるクラスを定義しています。

Datasetクラスは必ず 3つのメソッド を実装する必要があります。

① __init__() : 初期化

    ⇒クラスを呼び出した時に最初に実行される処理です。受け取ったデータを self.〇〇 に保存しておきます。

② __len__() : データ数を返す

    ⇒データが何枚あるかを返します。PyTorchが内部で使用します。

③ __getitem__() : 画像を1枚取り出す

    ⇒idx（インデックス）を受け取って、その番号の画像を1枚返します。

## 4. データ分割

In [8]:
train_paths_split, val_paths_split, y_train, y_val = train_test_split(
    train_paths, labels,    # train_pathsとlabelsを対応させて分割
    test_size=0.2,          # データの20%を検証用に分割
    random_state=42,        # 乱数シードを固定して再現性を確保
    stratify=labels         # ラベルの分布を保ったまま分割
)

print(f"train数     : {len(train_paths_split)}")
print(f"validation数: {len(val_paths_split)}")

train数     : 48000
validation数: 12000


stratify=labels

    ⇒クラスの比率を保ったまま分割する

stratifyなしの場合（偏りが出る可能性）

    train      : クラスA 90%, クラスB 10%

    validation : クラスA 50%, クラスB 50%

stratifyありの場合（比率が保たれる）

    train      : クラスA 80%, クラスB 20%

    validation : クラスA 80%, クラスB 20%

分割後のデータ構造

    train_paths_split  # 学習用の画像パス一覧

    val_paths_split    # 検証用の画像パス一覧

    y_train            # 学習用のラベル一覧

    y_val              # 検証用のラベル一覧

## 5. DataLoaderの作成

In [9]:
# VRAMに余裕がある為、バッチサイズを32→128に増量
#BATCH_SIZE = 32
BATCH_SIZE = 128

train_loader = DataLoader(
    HandwritingDataset(train_paths_split, y_train, train_transform),    # データセットを作成
    batch_size=BATCH_SIZE,      # バッチサイズを指定 128枚ずつ処理する
    shuffle=True                # データをシャッフルして学習の偏りを減らす
)

val_loader = DataLoader(
    HandwritingDataset(val_paths_split, y_val, test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    HandwritingDataset(test_paths, transform=test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"train   バッチ数: {len(train_loader)}")
print(f"val     バッチ数: {len(val_loader)}")
print(f"test    バッチ数: {len(test_loader)}")

train   バッチ数: 375
val     バッチ数: 94
test    バッチ数: 79


Dataset               DataLoader

「1枚の取り出し方」を定義   →  「何枚まとめて取り出すか」を定義

　        　　　　　　  　　

Image.open()         　　　　　　　　　　batch_size=32

transform適用        　　　　　　　　　shuffle=True

ラベルと紐付け       　　　　　　　　まとめてGPUに送る

## 6. モデルの定義

In [10]:
# クラス数を定義
NUM_CLASSES = len(np.unique(labels))

# GPUが利用可能ならGPUを、そうでなければCPUを使用する
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

print(f"クラス数: {NUM_CLASSES}")
print(f"使用デバイス: {device}")

クラス数: 10
使用デバイス: cuda


models.resnet18(weights="IMAGENET1K_V1")

    ⇒ImageNetで事前学習済みのResNet-18をダウンロードして読み込む

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    ⇒ResNet-18の元の最終層はImageNetの1000クラス分類用になっている

    なので最終層だけ、今回のクラス数(NUM_CLASSES)に付け替えることで、

    それ以前の層が学習した画像の特徴抽出はそのまま活用できる

## 7. 損失関数、オプティマイザの定義

In [11]:
criterion = nn.CrossEntropyLoss()

# バッチサイズを32→128に増量したため、学習率を比率に応じて増量
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4 * (128 / 32))

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

criterion = nn.CrossEntropyLoss()

    ⇒クロスエントロピー交差

    予測がどれだけ外れているかを数値化する損失関数です。

正解: クラス3

予測: クラス3の確率 0.9 → 損失小さい（良い予測）

予測: クラス3の確率 0.1 → 損失大きい（悪い予測）

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    ⇒損失を小さくするためにモデルのパラメータをどう更新するかを決めるオプティマイザ

lr=1e-4 は学習率（learning rate）= 0.0001

学習率が大きい → 大きく更新 → 学習が速いが不安定

学習率が小さい → 小さく更新 → 学習が遅いが安定

Adam は学習率を自動調整してくれる優秀なオプティマイザで、現在最もよく使われる

 scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

    ⇒学習率を自動で下げていくスケジューラ

学習が進むにつれて学習率を下げることで、細かい調整ができるようになり精度が上がりやすくなる

## 8. 学習のループ

In [12]:
EPOCHS = 10
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    train_loss, train_correct = 0, 0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()                                           # 前のバッチの勾配をリセット
        outputs = model(imgs)                                           # 予測                      
        loss = criterion(outputs, lbls)                                 # 損失を計算
        loss.backward()                                                 # 勾配を計算（逆伝播）
        optimizer.step()                                                # パラメータを更新
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == lbls).sum().item()       # 最も確率が高いクラスを予測値とする、正解ラベルと比較、正解数を合計

    # --- validation ---
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == lbls).sum().item()

    train_acc = train_correct / len(train_paths_split)
    val_acc   = val_correct   / len(val_paths_split)

    print(f"Epoch [{epoch+1}/{EPOCHS}]"
          f"  Train Loss: {train_loss/len(train_loader):.4f}  Train Acc: {train_acc:.4f}"
          f"  Val Loss: {val_loss/len(val_loader):.4f}  Val Acc: {val_acc:.4f}")

    # ベストモデルの保存
    # ACC(正解率)が最も高かった時点のモデルを保存する
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "../models/best_resnet18.pth")
        print(f"  → ベストモデルを保存 (Val Acc: {val_acc:.4f})")

    scheduler.step()

Epoch [1/10]  Train Loss: 0.0657  Train Acc: 0.9809  Val Loss: 0.0416  Val Acc: 0.9863
  → ベストモデルを保存 (Val Acc: 0.9863)
Epoch [2/10]  Train Loss: 0.0272  Train Acc: 0.9919  Val Loss: 0.0262  Val Acc: 0.9920
  → ベストモデルを保存 (Val Acc: 0.9920)
Epoch [3/10]  Train Loss: 0.0261  Train Acc: 0.9919  Val Loss: 0.0354  Val Acc: 0.9898
Epoch [4/10]  Train Loss: 0.0194  Train Acc: 0.9940  Val Loss: 0.0305  Val Acc: 0.9906
Epoch [5/10]  Train Loss: 0.0191  Train Acc: 0.9942  Val Loss: 0.0274  Val Acc: 0.9921
  → ベストモデルを保存 (Val Acc: 0.9921)
Epoch [6/10]  Train Loss: 0.0096  Train Acc: 0.9972  Val Loss: 0.0148  Val Acc: 0.9958
  → ベストモデルを保存 (Val Acc: 0.9958)
Epoch [7/10]  Train Loss: 0.0064  Train Acc: 0.9983  Val Loss: 0.0150  Val Acc: 0.9958
Epoch [8/10]  Train Loss: 0.0051  Train Acc: 0.9987  Val Loss: 0.0142  Val Acc: 0.9960
  → ベストモデルを保存 (Val Acc: 0.9960)
Epoch [9/10]  Train Loss: 0.0045  Train Acc: 0.9990  Val Loss: 0.0142  Val Acc: 0.9958
Epoch [10/10]  Train Loss: 0.0042  Train Acc: 0.9991  Val

＜ 全体の流れ ＞

EPOCHSの回数だけ繰り返す

　↓

【trainフェーズ】全trainデータで学習

　↓

【validationフェーズ】val dataで精度確認

　↓

ベストモデルなら保存

　↓

学習率を更新（scheduler）

    model.train()

⇒モデルを学習モードに。ドロップアウトなどが有効になる。

    model.eval()

⇒勾配の計算をしない。validationは学習しないので不要 → メモリ節約・高速化


学習中のGPU専用メモリが2.5GBしか使用していないのでバッチサイズを大きくしても大丈夫か？

    ⇒128で目安がVRAM65GB、256でVRAM8GB前後なので1228にしてもよい

    　しかし、勾配の更新回数が減ってしまうため、過学習しやすくなる

とても正解率が高いが過学習してない？

    ⇒過学習している場合は、TrainAccとValAccに乖離がある

    　今回はほぼ同じ数値なので過学習していないと判断できる

## 9.推論・提出ファイルの作成

In [13]:
# ベストモデルを読み込む
model.load_state_dict(torch.load("../models/best_resnet18.pth"))
model.eval()    # 評価モードに切り替える

predictions = []

with torch.no_grad():   # 勾配計算を無効化して推論を高速化
    for imgs in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        predictions.extend(outputs.argmax(1).cpu().numpy()) # 最もスコアが高いクラスを予測値とする ※argmax(1)とargmax(-1)は同じ意味

print(f"予測数: {len(predictions)}")
print(f"予測例: {predictions[:5]}")

C:\Users\youto\AppData\Local\Temp\ipykernel_48848\3828171330.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../models/best_resnet18.pt

予測数: 10000
予測例: [np.int64(7), np.int64(2), np.int64(1), np.int64(0), np.int64(4)]


In [14]:
# 数値順に並び替える
test_paths = sorted(
    collect_image_paths("../data/test/"),
    key=lambda p: int(p.stem.split("_")[1])
)

# 確認
for p in test_paths[:5]:
    print(p.name)

test_0.jpg
test_1.jpg
test_2.jpg
test_3.jpg
test_4.jpg


In [16]:
submit_df = pd.DataFrame({
    "file_name"  : [p.name for p in test_paths],
    "category_id": predictions
})

print(submit_df.head(10))
submit_df.to_csv("../submission.csv", index=False, header=False)
print("submission.csv を出力しました")

    file_name  category_id
0  test_0.jpg            7
1  test_1.jpg            2
2  test_2.jpg            1
3  test_3.jpg            0
4  test_4.jpg            4
5  test_5.jpg            1
6  test_6.jpg            4
7  test_7.jpg            9
8  test_8.jpg            5
9  test_9.jpg            9
submission.csv を出力しました


In [17]:
# 予測クラスの分布を確認
counter = collections.Counter(predictions)
for cls, count in sorted(counter.items()):
    print(f"クラス{cls}: {count}枚")

クラス0: 982枚
クラス1: 1139枚
クラス2: 1033枚
クラス3: 1013枚
クラス4: 980枚
クラス5: 890枚
クラス6: 956枚
クラス7: 1025枚
クラス8: 971枚
クラス9: 1011枚
